In [111]:
# This code is part of QCMet.
# 
# (C) Copyright 2024 National Physical Laboratory and National Quantum Computing Centre 
# 
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
# 
#      http://www.apache.org/licenses/LICENSE-2.0
# 
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License

In [27]:
import pathlib
import os
import numpy as np
import pygsti
from pygsti.modelpacks import smq1Q_XYI, smq2Q_XYCNOT
from pygsti.report import reportables as rptbl

#### Load in results from GST

In [28]:
root_dir = pathlib.Path(os.path.abspath(""))
# The following data has no noise on 1q operations so GST should predict perfect outcomes
gst_data_path = root_dir / "gst_data"
results = pygsti.io.read_results_from_dir(gst_data_path, name="StandardGST")
result_estimates = results.estimates["CPTPLND"].models["stdgaugeopt"]

In [29]:
n_qubits = np.log2(results.estimates['CPTPLND'].models['target']['rho0'].hilbert_schmidt_size)/2
if n_qubits == 1:
    model = smq1Q_XYI
elif n_qubits == 2:
    model = smq2Q_XYCNOT

ideal_model = model.target_model()
meas_projectors = ideal_model.effects

#### Calculate gate based execution metrics

In [30]:
gates = model.gates

if n_qubits == 1:
    basis = pygsti.baseobjs.Basis.cast("pp", 4)  # 1-qubit Pauli basis (2x2 matrices)
elif n_qubits == 2:
    basis = pygsti.baseobjs.Basis.cast("pp", 16)  # 2-qubit Pauli basis (2x2 matrices)

print("Gate execution quality metrics:")
for gate in gates:
    print(
        f"Process Fidelity Gate:{gate} = ",
        float(
            rptbl.entanglement_fidelity(
                ideal_model[gate], result_estimates[gate], basis
            )
        ),
    )
    print(
        f"Diamond Norm of Gate:{gate} = ",
        float(
            rptbl.half_diamond_norm(ideal_model[gate], result_estimates[gate], basis)
        ),
        "\n",
    )

print("\nState preparation and meausrement fidelity:")
print(
    "Fidelity of initial density matrix = ",
    float(rptbl.vec_fidelity(ideal_model["rho0"], result_estimates["rho0"], basis)),
)
print(
    rf"Measurement Fidelity  = ",
    float(1 - rptbl.povm_entanglement_infidelity(ideal_model, result_estimates, 'Mdefault'))
)

Gate execution quality metrics:
Process Fidelity Gate:('Gxpi2', 1) =  0.9871354892375207


Diamond Norm of Gate:('Gxpi2', 1) =  0.061272305784890325 

Process Fidelity Gate:('Gypi2', 1) =  0.9851290862482724
Diamond Norm of Gate:('Gypi2', 1) =  0.05990017716112636 

Process Fidelity Gate:('Gxpi2', 0) =  0.9957971640490499
Diamond Norm of Gate:('Gxpi2', 0) =  0.0111499137827521 

Process Fidelity Gate:('Gypi2', 0) =  0.9941629817134692
Diamond Norm of Gate:('Gypi2', 0) =  0.017088151372611677 

Process Fidelity Gate:('Gcnot', 0, 1) =  0.8858400478215285
Diamond Norm of Gate:('Gcnot', 0, 1) =  0.2063626977166566 


State preparation and meausrement fidelity:
Fidelity of initial density matrix =  0.8790047916021153
Measurement Fidelity  =  0.8750511362530655
